# Football Match Prediction Model Training

This notebook walks through training machine learning models to predict football match outcomes. We'll train three models:
- **Match Result**: Home Win, Draw, or Away Win
- **Over/Under 2.5 Goals**: Whether total goals will be over or under 2.5
- **Both Teams to Score (BTTS)**: Whether both teams will score

We'll use Python, pandas, scikit-learn, and joblib. At the end, models are saved as `.pkl` files for the Streamlit app.

In [ ]:
import pandas as pd
import numpy as np
import os


## Step 1: Load the Data

We load the engineered dataset that contains match features and target labels.

In [ ]:
# Load the engineered dataset
df = pd.read_csv('data/dataset_features.csv')
print('Shape:', df.shape)
print('Columns:', list(df.columns)[:10], '...')
print(df.head())

## Step 2: Explore Target Variables

The dataset has a `Target` column for match result (H=Home Win, D=Draw, A=Away Win). For other tasks, we'll create binary targets from the original match data.

In [ ]:
# Check target distribution
target_counts = df['Target'].value_counts()
print('Target distribution:')
display(target_counts)
print('Class names: H=Home Win, D=Draw, A=Away Win')

## Step 3: Prepare Features (X) and Target (y)

We'll use the 19 feature columns (excluding Team names, Date, and Target).

In [ ]:
# Separate features and target
X = df.drop(columns=['HomeTeam', 'AwayTeam', 'Date', 'Target'])
y = df['Target']
print('Features shape:', X.shape)
print('Target shape:', y.shape)
print(X.columns.tolist()]

## Step 4: Train-Test Split

We split the data into training (80%) and testing (20%) sets. This ensures the model generalizes to unseen matches.

In [ ]:
from sklearn.model_selection import train_test_split

# Split with random state for reproducibility
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print('Training set size:', X_train.shape[0])
print('Test set size:', X_test.shape[0])
print(y_train.value_counts())
print(y_test.value_counts())

## Step 5: Scale the Features

Gradient Boosting works well with scaled data. We'll use StandardScaler to standardize features (mean=0, std=1).

In [ ]:
from sklearn.preprocessing import StandardScaler

# Initialize scaler
scaler = StandardScaler()

# Fit on training data, transform both train and test
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X.columns)
print('Scaling complete. Mean after scaling:', round(X_train_scaled.mean().tolist(), 4))

## Step 6: Train the Match Result Model

We train a Gradient Boosting Classifier to predict Home Win (H), Draw (D), or Away Win (A).

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report

# Initialize model
model = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

# Train on scaled training data
model.fit(X_train_scaled, y_train)

# Predict on test set
y_pred = model.predict(X_test_scaled)

# Evaluate
accuracy = accuracy_score(y_test, y_pred)
print('Test Accuracy:', round(accuracy, 4))
print(classification_report(y_test, y_pred))

## Step 7: Train Over/Under 2.5 Goals Model

For this binary classification, we create a target: 1 = Over 2.5 goals, 0 = Under 2.5 goals.

In [ ]:
# Create binary target for over/under 2.5
y_ou = (df['FTHG'] + df['FTAG'] > 2).astype(int)  # 1 if over 2.5, 0 if under
print('Over/Under target distribution:')
print(y_ou.value_counts())

# Split
X_train, X_test, y_train_ou, y_test_ou = train_test_split(
    X, y_ou, test_size=0.2, random_state=42, stratify=y_ou
)

# Scale (using same scaler or new one - here we use a new scaler for clarity)
scaler_ou = StandardScaler()
X_train_ou_scaled = scaler_ou.fit_transform(X_train)
X_test_ou_scaled = scaler_ou.transform(X_test)

# Train model
model_ou = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)
model_ou.fit(X_train_ou_scaled, y_train_ou)

# Predict and evaluate
y_ou_pred = model_ou.predict(X_test_ou_scaled)
accuracy_ou = accuracy_score(y_test_ou, y_ou_pred)
print('Over/Under 2.5 Accuracy:', round(accuracy_ou, 4))

## Step 8: Train BTTS (Both Teams to Score) Model

Binary target: 1 = Both teams score, 0 = At least one team doesn't score.

In [ ]:
# Create BTTS target: 1 if both teams scored, 0 otherwise
y_btts = ((df['FTHG'] > 0) & (df['FTAG'] > 0)).astype(int)
print('BTTS target distribution:')
print(y_btts.value_counts())

# Split
X_train, X_test, y_train_btts, y_test_btts = train_test_split(
    X, y_btts, test_size=0.2, random_state=42, stratify=y_btts
)

# Scale
scaler_btts = StandardScaler()
X_train_btts_scaled = scaler_btts.fit_transform(X_train)
X_test_btts_scaled = scaler_btts.transform(X_test)

# Train model
model_btts = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)
model_btts.fit(X_train_btts_scaled, y_train_btts)

# Predict and evaluate
y_btts_pred = model_btts.predict(X_test_btts_scaled)
accuracy_btts = accuracy_score(y_test_btts, y_btts_pred)
print('BTTS Accuracy:', round(accuracy_btts, 4))

## Step 9: Save Models and Preprocessors

Now we save all trained models, scalers, and feature names using joblib. This is exactly what the Streamlit app uses for predictions.

In [ ]:
import joblib

# Save result model and companions
joblib.dump(model, 'models/best_match_predictor_model.pkl')
joblib.dump(scaler, 'models/scaler_result.pkl')
joblib.dump(list(X.columns), 'models/feature_names_result.pkl')
joblib.dump('placeholder', 'models/label_encoder_result.pkl')

# Save Over/Under model and companions
joblib.dump(model_ou, 'models/over_under_model.pkl')
joblib.dump(scaler_ou, 'models/scaler_ou.pkl')

# Save BTTS model and companions
joblib.dump(model_btts, 'models/btts_model.pkl')
joblib.dump(scaler_btts, 'models/scaler_btts.pkl')
joblib.dump(list(X.columns), 'models/feature_names_btts.pkl')

# Verify files were saved
model_files = [f for f in os.listdir('models/') if f.endswith('.pkl')]
print('Saved model files:')
print(model_files]

## Step 10: Load and Test a Saved Model (Optional)

You can load a saved model and make a prediction on new data, just like the Streamlit app does.

In [ ]:
# Example: Load the result model and make a prediction
# from utils import load_models
# models = load_models()
# model_res, le_res, scaler_res, feat_names_res = models['result']
# print('Model loaded successfully!')
# print('Features:', feat_names_res[:5], '...')
# print('Classes:', le_res.classes_)

## Training Complete!

All three models are now trained and saved. The Streamlit app (app.py) loads these .pkl files to make live predictions on match data.

**Next steps:**
- Run `streamlit run app.py` to use the prediction app
- Add more features (shots on target, possession, etc.) for better accuracy
- Train on more historical data for improved performance